# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Using exactly the **Feature** bucket from my ML-04 data contract — nothing from Label/proxy, nothing from Context, nothing from Excluded. Two things the ML-04 audit already found matter here: missingness in the keyword-context fields (`search_volume`, `competition`, `cpc`) and `word_count`/`char_count` tracks `content_type`, not randomly — so every field with real missingness gets a `has_<field>` flag *before* it gets filled, per the flyrank-data warning against a blind `fillna(0)`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
pool = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()
pool["tier_median_ctr"] = pool.groupby("position_tier")["ctr"].transform("median")
pool["is_opportunity"] = (pool.ctr < pool.tier_median_ctr).astype(int)  # ML-04's label, kept aside

numeric_feats = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "content_age_days", "days_since_last_update", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "ai_sessions_90d", "sessions_90d", "pageviews_90d", "users_90d",
    "engaged_sessions_90d", "scroll_events_90d", "days_with_impressions", "days_with_sessions",
]
cat_feats = [
    "content_type", "main_intent", "competition_level",
    "word_count_tier", "char_count_tier", "age_tier", "freshness_tier",
]

X = pool[numeric_feats + cat_feats + ["content_id", "client_id", "is_opportunity"]].copy()

# Flag real missingness BEFORE filling — these are the fields ML-04 confirmed are patterned by content_type.
missing_prone = ["search_volume", "competition", "cpc", "word_count", "char_count"]
for col in missing_prone:
    X[f"has_{col}"] = X[col].notna().astype(int)
    X[col] = X[col].fillna(X[col].median())

# Catch any other numeric column with stray missingness the same way.
for col in numeric_feats:
    if X[col].isna().any() and f"has_{col}" not in X.columns:
        X[f"has_{col}"] = X[col].notna().astype(int)
        X[col] = X[col].fillna(X[col].median())

X = pd.get_dummies(X, columns=cat_feats, drop_first=True)
feature_cols = [c for c in X.columns if c not in ("content_id", "client_id", "is_opportunity")]

print(f"Rows: {len(X)}")
print(f"Features: {len(feature_cols)}  (label base rate: {X['is_opportunity'].mean():.3f})")
print("has_-flags added:", [c for c in feature_cols if c.startswith("has_")])

Rows: 12023
Features: 43  (label base rate: 0.490)
has_-flags added: ['has_search_volume', 'has_competition', 'has_cpc', 'has_word_count', 'has_char_count', 'has_scroll_rate']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

All 25 features below are knowable from the page's own content/metadata or its already-elapsed 90-day traffic history — none of them require knowing this quarter's `ctr` or `clicks_90d` outcome, so all are legitimately "available before prediction."

In [2]:
notes = [
    ("search_volume", "monthly keyword search volume", "median-fill + has_ flag (missing for feedly article: not keyword-targeted)", "yes"),
    ("competition", "keyword competition score", "median-fill + has_ flag (same pattern as search_volume)", "yes"),
    ("cpc", "keyword cost-per-click estimate", "median-fill + has_ flag (same pattern)", "yes"),
    ("word_count", "page word count", "median-fill + has_ flag (28% missing for keyword article specifically)", "yes"),
    ("char_count", "page character count", "median-fill + has_ flag (mirrors word_count missingness)", "yes"),
    ("content_age_days", "days since the page was published", "none observed", "yes"),
    ("days_since_last_update", "days since last content edit", "none observed", "yes"),
    ("engagement_rate", "on-page engagement rate", "none observed (0 is a real value, per the ML-06 audit)", "yes"),
    ("scroll_rate", "share of sessions that scrolled", "median-fill + has_ flag (rare stray NaNs)", "yes"),
    ("ai_traffic_pct", "share of sessions from AI referrers", "none observed; can exceed 100 (cross-system rate, per flyrank-data)", "yes"),
    ("ai_sessions_90d", "session count from AI referrers, 90d", "none observed", "yes"),
    ("sessions_90d", "total sessions, 90d", "none observed", "yes"),
    ("pageviews_90d", "total pageviews, 90d", "none observed", "yes"),
    ("users_90d", "unique users, 90d", "none observed", "yes"),
    ("engaged_sessions_90d", "sessions meeting the engagement threshold, 90d", "none observed", "yes"),
    ("scroll_events_90d", "scroll events recorded, 90d", "none observed", "yes"),
    ("days_with_impressions", "days in the window with >=1 impression", "none observed", "yes"),
    ("days_with_sessions", "days in the window with >=1 session", "none observed", "yes"),
    ("content_type", "keyword article / feedly article / comparison article", "one-hot, drop_first", "yes"),
    ("main_intent", "search-intent category", "one-hot, drop_first", "yes"),
    ("competition_level", "low/medium/high bucket of competition", "one-hot, drop_first", "yes"),
    ("word_count_tier", "bucketed word_count", "one-hot, drop_first", "yes"),
    ("char_count_tier", "bucketed char_count", "one-hot, drop_first", "yes"),
    ("age_tier", "bucketed content_age_days", "one-hot, drop_first", "yes"),
    ("freshness_tier", "bucketed days_since_last_update", "one-hot, drop_first", "yes"),
]
notes_df = pd.DataFrame(notes, columns=["feature", "meaning", "missing_handling", "available_before_prediction"])
pd.set_option("display.max_colwidth", 80)
notes_df

,feature,meaning,missing_handling,available_before_prediction
0,search_volume,monthly keyword search volume,median-fill + has_ flag (missing for feedly article: not keyword-targeted),yes
1,competition,keyword competition score,median-fill + has_ flag (same pattern as search_volume),yes
2,cpc,keyword cost-per-click estimate,median-fill + has_ flag (same pattern),yes
3,word_count,page word count,median-fill + has_ flag (28% missing for keyword article specifically),yes
4,char_count,page character count,median-fill + has_ flag (mirrors word_count missingness),yes
5,content_age_days,days since the page was published,none observed,yes
6,days_since_last_update,days since last content edit,none observed,yes
7,engagement_rate,on-page engagement rate,"none observed (0 is a real value, per the ML-06 audit)",yes
8,scroll_rate,share of sessions that scrolled,median-fill + has_ flag (rare stray NaNs),yes
9,ai_traffic_pct,share of sessions from AI referrers,"none observed; can exceed 100 (cross-system rate, per flyrank-data)",yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Test 1 — deliberately smuggle in a label-derived feature and watch the score jump.** Per the skill's own verification method: if adding `ctr` back in doesn't blow the score up, my test harness itself is broken.

In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

def fit_eval(frame, cols, train_idx, test_idx, label):
    Xtr, Xte = frame.iloc[train_idx][cols], frame.iloc[test_idx][cols]
    ytr, yte = frame.iloc[train_idx]["is_opportunity"], frame.iloc[test_idx]["is_opportunity"]
    scaler = StandardScaler()
    clf = LogisticRegression(max_iter=2000)
    clf.fit(scaler.fit_transform(Xtr), ytr)
    auc = roc_auc_score(yte, clf.predict_proba(scaler.transform(Xte))[:, 1])
    print(f"{label:32s} AUC={auc:.3f}  base_rate(test)={yte.mean():.3f}  n_test={len(yte)}")
    return auc, clf

# Grouped split: hold out whole clients, never split a client's pages across train/test.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, groups=X["client_id"]))

auc_honest, honest_clf = fit_eval(X, feature_cols, train_idx, test_idx, "HONEST (grouped split)")

X["ctr_LEAK"] = pool["ctr"].values  # deliberately reintroducing the label's own source column
auc_leaky, _ = fit_eval(X, feature_cols + ["ctr_LEAK"], train_idx, test_idx, "LEAKY (ctr sneaked in)")
print(f"\n-> Confession: AUC jumps {auc_honest:.3f} to {auc_leaky:.3f} the moment ctr is added. "
      "Test harness confirmed working; ctr correctly stays OUT of the real feature set.")

# Test 2: grouped split vs random (ungrouped) split -- the gap is itself a finding.
Xtr2, Xte2, ytr2, yte2 = train_test_split(
    X[feature_cols], X["is_opportunity"], test_size=0.25, random_state=42, stratify=X["is_opportunity"]
)
scaler2 = StandardScaler()
clf2 = LogisticRegression(max_iter=2000).fit(scaler2.fit_transform(Xtr2), ytr2)
auc_random = roc_auc_score(yte2, clf2.predict_proba(scaler2.transform(Xte2))[:, 1])
print(f"\nrandom split AUC={auc_random:.3f}  vs  grouped split AUC={auc_honest:.3f}")
print(f"gap = {auc_random - auc_honest:+.3f} -- small, so client memorization isn't doing much "
      "heavy lifting here (a bigger gap would mean the model was leaning on which client a page "
      "belongs to rather than the page's own signal).")

# Test 3: sanity-check the top coefficients -- "too good" gets investigated, not celebrated.
coefs = pd.Series(honest_clf.coef_[0], index=feature_cols).sort_values(key=abs, ascending=False)
print("\ntop 5 |coefficient| in the honest model:")
print(coefs.head(5))
print(f"\ncorrelation(pageviews_90d, users_90d) = {pool['pageviews_90d'].corr(pool['users_90d']):.3f}")
print("-> RED FLAG, not a real signal: pageviews_90d and users_90d carry the two largest "
      "coefficients with opposite signs. They're near-collinear (corr above), so the model is "
      "fitting their unstable difference, not learning something meaningful about either one. "
      "This isn't leakage -- both are legitimately pre-prediction traffic history -- but it's an "
      "unstable pair that shouldn't be read as 'pageviews matter 3.7x more than search_volume.'")

HONEST (grouped split)           AUC=0.789  base_rate(test)=0.441  n_test=824
LEAKY (ctr sneaked in)           AUC=0.994  base_rate(test)=0.441  n_test=824

-> Confession: AUC jumps 0.789 to 0.994 the moment ctr is added. Test harness confirmed working; ctr correctly stays OUT of the real feature set.

random split AUC=0.768  vs  grouped split AUC=0.789
gap = -0.021 -- small, so client memorization isn't doing much heavy lifting here (a bigger gap would mean the model was leaning on which client a page belongs to rather than the page's own signal).

top 5 |coefficient| in the honest model:
pageviews_90d        -3.669180
users_90d             3.481443
days_with_sessions   -0.929135
search_volume         0.370349
scroll_events_90d    -0.326789
dtype: float64

correlation(pageviews_90d, users_90d) = 0.966
-> RED FLAG, not a real signal: pageviews_90d and users_90d carry the two largest coefficients with opposite signs. They're near-collinear (corr above), so the model is fitting their uns

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
excluded = {
    "ctr": "the label itself -- the outcome I'm trying to flag",
    "clicks_90d": "numerator inside ctr; using it lets the model see its own target",
    "trend_direction": "the pipeline's decline label is computed from this -- keeping it out avoids quietly redefining the task",
    "trend_pct": "same reason as trend_direction -- it IS the trend label's source",
    "impressions_last_30d": "exists only to build trend_pct -- excluded for the same reason as trend_pct",
    "clicks_last_30d": "same reason -- feeds trend_pct only",
    "sessions_last_30d": "same reason -- feeds trend_pct only",
    "impressions_prev_30d": "same reason -- feeds trend_pct only",
    "clicks_prev_30d": "same reason -- feeds trend_pct only",
    "sessions_prev_30d": "same reason -- feeds trend_pct only",
    "provider_used": "data dictionary marks this 'not a model feature' -- which LLM wrote the copy isn't actionable and risks provider bias",
    "model_used": "same reason as provider_used",
    "content_id": "identifier only -- for joins, never a learned input",
    "client_id": "identifier only -- used for the grouped split above, never a learned input",
    "avg_position": "defines the peer group (tier) the label compares within -- not a learned input",
    "position_tier": "same reason as avg_position -- defines the comparison group, not a feature",
    "impression_tier": "used to build the eligible pool (visibility filter), not a learned input",
    "impressions_90d": "same reason as impression_tier -- eligibility filter, not a learned input",
}
print(f"{len(excluded)} fields excluded:\n")
for field, why in excluded.items():
    print(f"  {field:24s} {why}")

overlap = set(excluded) & set(feature_cols)
print(f"\nExcluded fields present in the actual feature matrix: {overlap or 'none'}")
assert not overlap, "An excluded field leaked into the feature matrix."

18 fields excluded:

  ctr                      the label itself -- the outcome I'm trying to flag
  clicks_90d               numerator inside ctr; using it lets the model see its own target
  trend_direction          the pipeline's decline label is computed from this -- keeping it out avoids quietly redefining the task
  trend_pct                same reason as trend_direction -- it IS the trend label's source
  impressions_last_30d     exists only to build trend_pct -- excluded for the same reason as trend_pct
  clicks_last_30d          same reason -- feeds trend_pct only
  sessions_last_30d        same reason -- feeds trend_pct only
  impressions_prev_30d     same reason -- feeds trend_pct only
  clicks_prev_30d          same reason -- feeds trend_pct only
  sessions_prev_30d        same reason -- feeds trend_pct only
  provider_used            data dictionary marks this 'not a model feature' -- which LLM wrote the copy isn't actionable and risks provider bias
  model_used           

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.